# tensor-to-device — worked example 3: .to(device, dtype) — move and cast in one call

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-to-device`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`.to(device=..., dtype=...)` combines a device move and a dtype cast into a single call, which is more efficient than doing them separately. This is useful when loading float64 data from NumPy (which defaults to float64) and converting to float32 for GPU training — one `.to('cuda', dtype=torch.float32)` handles both.

## Worked solution

**Step 1 — Separate calls are two operations.**
`x.to('cuda').to(torch.float32)` performs two memory allocations and two transfers. The intermediate `float64` CUDA tensor is created and immediately discarded.

**Step 2 — Combined call is one operation.**
`x.to(device='cuda', dtype=torch.float32)` fuses the two: the data is cast and transferred in one kernel.

**Step 3 — Common use case.**
When loading with NumPy or Pandas, data is typically `float64`. Neural networks usually run in `float32`. The combined `.to()` at data loading time is the clean solution.

**Step 4 — dtype alone.**
`x.to(dtype=t.float16)` works without a device argument — you can cast in place (for mixed-precision inference) without moving devices.

In [ ]:
import torch as t
import numpy as np

# Simulate loading float64 data from NumPy
np.random.seed(42)
data_np = np.random.randn(5, 3)  # float64 by default
print('numpy dtype:', data_np.dtype)   # float64

# Convert to torch float64
x_f64 = t.tensor(data_np)
print('tensor dtype:', x_f64.dtype)    # float64
print('tensor device:', x_f64.device)  # cpu

# Option A: two separate calls
device = 'cuda' if t.cuda.is_available() else 'cpu'
x_a = x_f64.to(device).to(t.float32)
print('Option A dtype:', x_a.dtype, '| device:', x_a.device)

# Option B: one combined call (preferred)
x_b = x_f64.to(device=device, dtype=t.float32)
print('Option B dtype:', x_b.dtype, '| device:', x_b.device)

print('Both equal:', t.allclose(x_a, x_b))  # True